# FPT linked statement development walkthrough

This notebook follows the opening statement, explicit Q4 stub, five annual forecasts,
cash/debt/equity schedules, economic challenges and cross-platform reconciliation.
Forward drivers are **documented analyst scenarios**, calibrated to verified opening facts and issuer updates available by23November2026. No target price is calculated.

The saved cells were evaluated sequentially in a plain-Python harness on Python 3.12.
Jupyter-kernel execution has not been verified on this host. Start a Python 3.10+
kernel from the repository or its notebooks directory to rerun the same cells.


In [1]:
from pathlib import Path
import json, sys
from copy import deepcopy
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/equity_valuation').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from equity_valuation.intake import validate_opening
from equity_valuation.three_statement import run_forecast
opening = json.loads((ROOT / 'data/intake/clock_c_opening_pending.json').read_text())
configs = json.loads((ROOT / 'configs/forecast_research.json').read_text())['scenarios']
print('Units:', opening['unit'])
print('Opening:', opening['period_end'], '| information cutoff:', opening['information_cutoff'])


Units: VND billion
Opening: 2026-09-30 | information cutoff: 2026-11-23


## 1 Opening data and source status

The input contains 18 groups from reported statement codes. Component accounting
checks are separate from original-file provenance. The downloaded HOSE binary was hashed and visually reviewed. Passing the accounting
identity must not silently approve the source or forward assumptions.


In [2]:
intake = validate_opening(opening)
print('Source intake:', intake['status'])
for name, difference in intake['reconciliations'].items():
    print(f'{name:25s} {difference:,.6f}')
print('Remaining issue types:', sorted({e['code'] for e in intake['errors']}))
print('Primary filing:', opening['source_url'])


Source intake: PASS
asset_components          0.000000
liability_components      0.000000
equity_components         0.000000
balance_sheet             0.000000
Remaining issue types: []
Primary filing: https://staticfile.hsx.vn/Uploads/UploadDocuments/2508914/20261028%20-%20FPT%20-%20BCTC%20hop%20nhat%20Quy%203%20nam%202026.pdf


## Reported-note calibration references

Ten historical accounting bridges separate receivables, deposit principal and
accrued interest, associates, debt/perimeter, remaining loan maturities, cash
flows and financial income/expense. Every raw amount is integer VND and has a
printed-page locator. Original binary provenance is recorded separately from arithmetic agreement.
These are historical calibration references; future drivers remain analyst judgements.


In [3]:
from equity_valuation.reported_schedules import analyse_reported_schedules
reported = json.loads((ROOT / 'data/intake/q3_2026_reported_schedules.json').read_text())
evidence = analyse_reported_schedules(reported)
assert evidence['all_accounting_bridges_pass']
for check in evidence['checks']:
    print(check['schedule'], 'difference VND:', check['difference_vnd'])
print('Historical associate dividend receipts / profit:',
      f"{evidence['metrics']['associate_received_dividends_to_profit']:.2%}")
print('Original binary verified:', evidence['source_verified'])
print('Important: FTEL perimeter movements and current-portion transfers are not cash repayments.')


receivables difference VND: 0
short_investments difference VND: 0
associates difference VND: 0
short_debt difference VND: 0
long_debt difference VND: 0
loan_maturity difference VND: 0
cfo difference VND: 0
cfi difference VND: 0
finance_income difference VND: 0
finance_expense difference VND: 0
Historical associate dividend receipts / profit: 9.73%
Original binary verified: True
Important: FTEL perimeter movements and current-portion transfers are not cash repayments.


## 2 Periods and documented analyst assumptions

Q4 2026 is a 92/365 stub, not an actual or full-year 2026 forecast. For following
years, working-capital balances use the annual run rate. Trade DSO drives gross trade only. Vendor advances, contract receivables,
other balances and long deposits have separate schedules. October management
figures are held fixed across cases; November and December are forecast.


In [4]:
print('Period       Annual run rate     EBIT margin      DSO     Capex ratio')
for p in configs['base']['periods']:
    print(f"{p['label']:12s} {p['annual_revenue']:14,.1f} {p['ebit_margin']:14.1%} {p['dso']:8.1f} {p['capex_ratio']:14.1%}")


Period       Annual run rate     EBIT margin      DSO     Capex ratio
Q4 2026E           59,536.6          14.8%     75.0          10.0%
2027E              61,623.1          15.0%     75.0           7.0%
2028E              68,411.2          15.2%     75.0           6.5%
2029E              75,284.8          15.5%     75.0           6.0%
2030E              82,119.8          15.8%     75.0           5.5%
2031E              87,957.9          16.0%     75.0           5.5%


## 3 One linked build for all three cases

Operating profit, tax, earnings attribution, noncash associate earnings,
dividends, working capital, capex and debt flows feed the same cash account.
Cash is a cash-flow result; equity is rolled from earnings and distributions.
An explicit facility commitment can leave an infeasible funding gap.


In [5]:
results = {name: run_forecast(opening, cfg, development=True) for name, cfg in configs.items()}
print('Case       Last cash     Last debt    Last CFO    Max accounting difference')
for name, result in results.items():
    last = result['periods'][-1]
    debt = last['balance_sheet']['short_term_debt'] + last['balance_sheet']['long_term_debt']
    difference = max(abs(v) for row in result['periods'] for v in row['checks'].values())
    print(f"{name:10s} {last['balance_sheet']['cash']:11,.1f} {debt:13,.1f} {last['cash_flow']['cfo']:11,.1f} {difference:24.10f}")


Case       Last cash     Last debt    Last CFO    Max accounting difference
base          28,650.9      13,824.3    13,388.4             0.0000000000
downside       3,117.2      10,162.1     8,235.7             0.0000000000
upside        39,376.5      13,824.3    18,568.3             0.0000000000


## 4 Working capital challenge

Increase receivable days by 30 in the first period. The incremental cash use
must equal full-year sales times30/365, adjusted for the tax effect of incremental
receivable provisions. The provision charge changes profit; nontrade balances
are unchanged.. The financing or cash consequence is explicit.


In [6]:
stress = deepcopy(configs['base'])
stress['periods'][0]['dso'] += 30
stressed = run_forecast(opening, stress, development=True)['periods'][0]
base = results['base']['periods'][0]
p = stress['periods'][0]
expected_cash_use = p['wc_revenue'] * 30 / 365 * (1-p['allowance_ratio']*p['tax_rate'])
observed_cash_use = base['cash_flow']['cfo'] - stressed['cash_flow']['cfo']
assert abs(expected_cash_use-observed_cash_use) < 1e-6
print('Expected additional cash use:', round(expected_cash_use, 3))
print('Observed additional cash use:', round(observed_cash_use, 3))


Expected additional cash use: 4498.338
Observed additional cash use: 4498.338


## 5 Infeasible financing is not an accounting plug

A deliberately extreme receivable assumption and zero additional commitment
must surface a funding shortfall. Negative projected cash in this experiment
means the scenario cannot be financed under the specified policy.


In [7]:
failure = deepcopy(configs['base'])
failure['facility_limit'] = 0
failure['periods'][0]['dso'] = 700
row = run_forecast(opening, failure, development=True)['periods'][0]
assert row['funding']['status'] == 'INFEASIBLE'
print('Funding status:', row['funding']['status'])
print('Cash-floor shortfall:', round(row['funding']['funding_gap'], 3))
print('Balance-sheet difference:', round(row['checks']['balance_sheet'], 8))


Funding status: INFEASIBLE
Cash-floor shortfall: 88321.838
Balance-sheet difference: 0.0


## 6 Excel and browser reconciliation

Excel independently calculates the saved cases using formulas. The HTML model
is separately implemented in JavaScript. These checks support mathematical
agreement, not approval of assumptions or an investment recommendation.


In [8]:
out = ROOT / 'outputs/fpt_development'
excel = json.loads((out / 'excel_python_reconciliation.json').read_text())
browser = json.loads((out / 'dashboard_reconciliation.json').read_text())
assert all(abs(r['difference']) < 1e-6 for r in excel['results'])
assert browser['all_pass']
print('Excel comparison points:', excel['checks'])
print('Browser comparison points:', browser['numeric_comparisons'])
print('Valuation published:', results['base']['valuation_published'])


Excel comparison points: 378
Browser comparison points: 1584
Valuation published: False


## 7 Stress captures and source/forecast boundary

The source receipt records the downloaded HOSE PDF digest and visually reviewed
pages. The assumption register documents every numeric period driver, including
analyst judgement and legacy inactive fields. It does not claim verification of
future outcomes. Stress captures are recalculated by the pipeline.


In [9]:
stress_results = json.loads((ROOT / 'artifacts/research_stresses/results.json').read_text())['results']
print('Stress case               Minimum cash     Largest funding gap')
for name, item in stress_results.items():
    minimum = min(p['balance_sheet']['cash'] for p in item['periods'])
    gap = max(p['funding']['funding_gap'] for p in item['periods'])
    print(f'{name:28s} {minimum:12,.1f} {gap:22,.1f}')
assert stress_results['no_revolver_refinancing']['periods'][0]['funding']['status'] == 'INFEASIBLE'
print('A gap is relative to the fixed-deposit, zero-additional-facility policy; it is not an issuer insolvency claim.')


Stress case               Minimum cash     Largest funding gap
base                              7,619.0                    0.0
downside                            383.1                3,772.3
upside                            9,512.7                    0.0
dso_plus_30                       3,150.7                    0.0
rates_plus_200bp                  7,566.9                    0.0
no_revolver_refinancing          -6,205.2                8,956.7
interest_collection_delayed       7,125.8                    0.0
no_associate_dividends            7,369.0                    0.0
combined_stress                  -9,197.4               12,617.9
A gap is relative to the fixed-deposit, zero-additional-facility policy; it is not an issuer insolvency claim.


## 8 Scope before valuation

The analyst model separates trade/nontrade receivables, provisions, prepaid
stocks, operating liabilities, deposit principal/accrual/receipts, term and
revolving debt, current/noncurrent classification, CIP and associate dividends.
Tax timing, capital mix, useful lives, refinancing, FX and dividend policies
remain explicit modelling judgements and limitations. Read the full assumption
register and `docs/PREVALUATION_REVIEW.md` before connecting a valuation.
No target price or academic findings report is generated here. No independent
human review, native Excel recalculation or Jupyter-kernel run is claimed.
